# BI - LSTM MODEL Tonal Analysis For Voice-Based Sentiment Analysis

In [15]:
import torch
from torch import nn
from dataset import MFCCData
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, f1_score

In [26]:
# Available in models.py
class BiLSTM(nn.Module):
    def __init__(self, input_dim = 40, hidden_dim = 128, num_layers=2, output_dim=3, dropout=0.3):
        super(BiLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        
        self.fc = nn.Linear(hidden_dim * 2, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        pool = torch.mean(lstm_out, dim=1)
        out = self.dropout(pool) 
        out = self.fc(out)
        return out

In [27]:
def train(device, model, dataloader, criterion, optimizer):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    
    for mfcc, labels in dataloader:
        mfcc, labels = mfcc.to(device), labels.to(device)
        
        optimizer.zero_grad()
        output = model(mfcc)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(output, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(dataloader)
    return avg_loss, f1, acc
    

In [28]:
def eval(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for mfcc, label in dataloader:
            mfcc, label = mfcc.to(device), label.to(device)
            output = model(mfcc)
            loss = criterion(output, label)
            total_loss += loss
            
            preds = torch.argmax(output, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(label.cpu().numpy())
            
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    avg_loss = total_loss / len(dataloader)
    return avg_loss, f1, acc

In [29]:
labels_map = {
    "neutral": 0,
    'joy': 1,
    "surprise": 1,
    'sadness': 2,
    'anger': 2,
    'disgust': 2,
    'fear': 2
}
dataset = MFCCData(data='MELD.Raw/train_sent_emo.csv', aud='MELD.Wav/train/', label=labels_map)

In [30]:
# Set your split lengths
total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

training, validation = random_split(dataset, [train_size, val_size])


In [37]:
def collate(batch):
    from torch.nn.utils.rnn import pad_sequence
    mfccs, labels = zip(*batch)
    
    # No need to re-wrap if they're already tensors
    padded_mfccs = pad_sequence(mfccs, batch_first=True)  # Shape: (batch, time, features)
    labels = torch.tensor(labels)
    return padded_mfccs, labels

In [38]:
train_loader = DataLoader(training, batch_size=32, shuffle=True, collate_fn=collate)
val_loader = DataLoader(validation, batch_size=32, shuffle=False, collate_fn=collate)


In [42]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiLSTM().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)
criterion = nn.CrossEntropyLoss()

best_val_f1 = 0
patience = 5
patience_counter = 0

num_epochs = 100

for epoch in range(num_epochs):
    train_loss, train_acc, train_f1 = train(model=model, dataloader=train_loader, optimizer=optimizer, criterion=criterion, device=device)
    val_loss, val_acc, val_f1 = eval(model=model, dataloader=val_loader, criterion=criterion, device=device)
    
    scheduler.step(val_f1)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    print("-" * 50)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_bilstm_model.pt")
        print("🔥 Best model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("⏹️ Early stopping triggered.")
            break

/home/kurty/Project/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/100
Train Loss: 1.0390 | Acc: 0.3729 | F1: 0.4787
Val   Loss: 1.0265 | Acc: 0.4486 | F1: 0.4905
--------------------------------------------------
🔥 Best model saved!
Epoch 2/100
Train Loss: 1.0185 | Acc: 0.4209 | F1: 0.5007
Val   Loss: 1.0133 | Acc: 0.4414 | F1: 0.5060
--------------------------------------------------
🔥 Best model saved!
Epoch 3/100
Train Loss: 1.0054 | Acc: 0.4412 | F1: 0.5087
Val   Loss: 1.0114 | Acc: 0.4208 | F1: 0.5060
--------------------------------------------------
Epoch 4/100
Train Loss: 0.9896 | Acc: 0.4616 | F1: 0.5192
Val   Loss: 1.0234 | Acc: 0.4235 | F1: 0.5065
--------------------------------------------------
🔥 Best model saved!
Epoch 5/100
Train Loss: 0.9716 | Acc: 0.4916 | F1: 0.5396
Val   Loss: 1.0145 | Acc: 0.4290 | F1: 0.4995
--------------------------------------------------
Epoch 6/100
Train Loss: 0.9451 | Acc: 0.5217 | F1: 0.5590
Val   Loss: 1.0334 | Acc: 0.4510 | F1: 0.5000
--------------------------------------------------
Epoch 7/10

In [41]:
# Created by DaBloat